# OCR CCCD & Giấy phép kinh doanh với Qwen3-VL-8B-Instruct (tách 2 nhóm file)

Notebook chạy trên **Google Colab**: đọc ảnh/PDF của **CCCD** và/hoặc **Giấy chứng nhận đăng ký doanh nghiệp / hộ kinh doanh (GPKD)**, trả về một JSON đúng cấu trúc.

Khác bản trước: client **tự chia file thành hai nhóm** `cccd_files` và `gpkd_files`, nên hệ thống không phải đoán loại giấy tờ. Mỗi nhóm được gọi model với một dòng gợi ý riêng, và chỉ được điền đúng các trường của nhóm đó.

**Các trường hợp hỗ trợ**
- Chỉ gửi `gpkd_files` (một GPKD) → ba trường CCCD là `null`, các trường còn lại lấy từ GPKD.
- Chỉ gửi `cccd_files` → chỉ ba trường CCCD có giá trị.
- Gửi **cả hai** → gộp thành một JSON.
- Không gửi file nào → API trả `400`.

**Chuẩn bị**
1. `Runtime` → `Change runtime type` → chọn GPU.
   - **T4 (miễn phí, 16GB)**: tự bật nén 4-bit để vừa bộ nhớ.
   - **L4 / A100 (Colab Pro)**: chạy bfloat16 đầy đủ, chính xác hơn.
2. Chạy lần lượt từng cell từ trên xuống.
3. Lần đầu tải model mất vài phút.

> CCCD là dữ liệu cá nhân nhạy cảm. Chỉ dùng ảnh mẫu đã che hoặc được phép sử dụng, và xoá file khi test xong.

## 1. Kiểm tra GPU

In [ ]:
import subprocess, torch

print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
    capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime > Change runtime type > chọn GPU"
print("Compute capability:", torch.cuda.get_device_capability())

## 2. Cài thư viện

Qwen3-VL cần `transformers >= 4.57.0`. Nếu sau khi cài mà import báo lỗi, vào `Runtime` → `Restart session` rồi chạy lại từ đầu. Các cảnh báo đỏ về xung đột phiên bản (`requests`, `pillow`, `opentelemetry`...) có thể bỏ qua.

In [ ]:
!pip install -q -U "transformers>=4.57.0" accelerate bitsandbytes pymupdf pillow pydantic fastapi uvicorn python-multipart requests

## 3. Cấu hình

In [ ]:
import torch

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

# GPU compute capability < 8 (vd T4) không có bfloat16 tốt -> dùng 4-bit.
# L4 / A100 (>= 8) -> chạy bfloat16 đầy đủ.
LOAD_IN_4BIT = torch.cuda.get_device_capability()[0] < 8

MAX_SIDE = 1600       # cạnh dài nhất của ảnh (px) trước khi đưa vào model
PDF_DPI = 150         # độ phân giải khi chuyển PDF thành ảnh
MAX_PDF_PAGES = 3     # số trang PDF tối đa đọc cho mỗi file
STRICT = False        # True: trường sai định dạng bị đặt null; False: giữ giá trị + cảnh báo

# Giới hạn upload cho API
MAX_FILE_MB = 15              # dung lượng tối đa mỗi file
MAX_FILES_PER_GROUP = 5       # số file tối đa trong cccd_files hoặc gpkd_files
ALLOWED_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff", ".pdf"}

print("MODEL_ID     :", MODEL_ID)
print("LOAD_IN_4BIT :", LOAD_IN_4BIT)

## 4. Tải model

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

kwargs = dict(device_map="auto")
if LOAD_IN_4BIT:
    kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
else:
    kwargs["dtype"] = torch.bfloat16

model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **kwargs)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model.eval()

print("Đã tải xong. VRAM đang dùng: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

## 5. Schema đầu ra

`GROUP_FIELDS` quy định nhóm file nào được phép điền trường nào:
- `CCCD` → `id_cccd`, `ho_ten_cccd`, `thuong_tru_cccd`
- `GPKD` → `loai_giay_to` và các trường doanh nghiệp (kể cả `id_cccd_dai_dien`)

In [ ]:
import json, re
from pydantic import BaseModel


class ExtractionResult(BaseModel):
    id_cccd: str | None = None
    ho_ten_cccd: str | None = None
    thuong_tru_cccd: str | None = None

    loai_giay_to: str | None = None
    ten_cong_ty: str | None = None
    dia_chi_tru_so: str | None = None
    ma_so_thue: str | None = None
    so_dien_thoai: str | None = None
    email: str | None = None
    nguoi_dai_dien: str | None = None
    id_cccd_dai_dien: str | None = None
    chuc_danh: str | None = None


FIELDS = list(ExtractionResult.model_fields.keys())
CCCD_FIELDS = ["id_cccd", "ho_ten_cccd", "thuong_tru_cccd"]
GPKD_FIELDS = [f for f in FIELDS if f not in CCCD_FIELDS and f != "loai_giay_to"]

GROUP_FIELDS = {
    "CCCD": CCCD_FIELDS,
    "GPKD": ["loai_giay_to"] + GPKD_FIELDS,
}

print(FIELDS)

## 6. Prompt

Vẫn dùng một prompt chung `PROMPT_BOTH`, nhưng khi gọi model sẽ **nối thêm một đoạn gợi ý theo nhóm file** (`HINTS`) vào cuối, để model biết chắc ảnh là CCCD hay GPKD. Prompt là phần ảnh hưởng độ chính xác nhiều nhất, bạn nên chỉnh theo ảnh thật của mình.

In [ ]:
PROMPT_BOTH = """Bạn là hệ thống OCR chuyên đọc thông tin từ giấy tờ Việt Nam, bao gồm:
- CCCD (Căn cước công dân) hoặc Căn cước
- Giấy chứng nhận đăng ký doanh nghiệp
- Giấy chứng nhận đăng ký hộ kinh doanh

Chỉ trả về JSON hợp lệ.
Không giải thích.
Không dùng Markdown.
Nếu một trường không hiển thị rõ ràng thì trả về null.
Giữ nguyên dấu tiếng Việt.

Trả về đúng cấu trúc JSON sau:
{
  "id_cccd": null,
  "ho_ten_cccd": null,
  "thuong_tru_cccd": null,

  "loai_giay_to": null,
  "ten_cong_ty": null,
  "dia_chi_tru_so": null,
  "ma_so_thue": null,
  "so_dien_thoai": null,
  "email": null,
  "nguoi_dai_dien": null,
  "id_cccd_dai_dien": null,
  "chuc_danh": null
}

========================
QUY TẮC CHUNG
========================

- Chỉ extract đúng thông tin có trên ảnh.
- Không tự suy luận dữ liệu không tồn tại rõ ràng.
- Không đoán và không bịa. Không chắc chắn thì trả về null.
- Tất cả mã số phải là chuỗi (string).
- KHÔNG bỏ số 0 đứng đầu.
- KHÔNG chuyển thành số nguyên.
- Giữ nguyên dấu tiếng Việt.
- Nhiều ảnh/trang thuộc cùng một giấy tờ: đọc toàn bộ rồi gộp thành MỘT kết quả.

========================
PHÂN BIỆT LOẠI ẢNH
========================

- Ảnh là thẻ CCCD/Căn cước:
  + Chỉ điền id_cccd, ho_ten_cccd, thuong_tru_cccd.
  + Tất cả trường còn lại (kể cả loai_giay_to) = null.

- Ảnh là giấy đăng ký doanh nghiệp / hộ kinh doanh:
  + Chỉ điền từ loai_giay_to trở xuống.
  + id_cccd, ho_ten_cccd, thuong_tru_cccd = null.
  + Số CCCD in trong giấy phép là của người đại diện, ghi vào id_cccd_dai_dien, KHÔNG ghi vào id_cccd.

1. QUY TẮC CCCD

- id_cccd là số định danh cá nhân (mục "Số / No."), gồm 12 chữ số.

- ho_ten_cccd là họ tên đầy đủ trên CCCD.

- thuong_tru_cccd là toàn bộ địa chỉ ở mục "Nơi thường trú" hoặc "Nơi cư trú".

- Địa chỉ có thể xuống nhiều dòng:
→ phải nối đầy đủ thành một chuỗi duy nhất.

- KHÔNG đưa các thông tin sau vào thuong_tru_cccd:
  + Giới tính
  + Quốc tịch
  + Ngày sinh
  + Quê quán
  + Số CCCD

- Địa chỉ đúng có dạng: "số nhà, đường, phường/xã, quận/huyện, tỉnh/thành phố".

2. QUY TẮC PHÂN LOẠI GPKD

Các giá trị hợp lệ cho loai_giay_to:
- "DOANH_NGHIEP"
- "HO_KINH_DOANH"
- null

Phân loại dựa vào TIÊU ĐỀ của giấy:
- Tiêu đề "GIẤY CHỨNG NHẬN ĐĂNG KÝ HỘ KINH DOANH" (mọi biến thể chữ hoa/thường)
→ loai_giay_to = "HO_KINH_DOANH".
- Tiêu đề "GIẤY CHỨNG NHẬN ĐĂNG KÝ DOANH NGHIỆP" hoặc tương đương
→ loai_giay_to = "DOANH_NGHIEP".
- Không phân loại dựa vào cụm từ xuất hiện trong phần nội dung.
- Nếu ảnh không phải giấy đăng ký doanh nghiệp/hộ kinh doanh thì loai_giay_to = null.

3. QUY TẮC GPKD

- ma_so_thue là mã số thuế/mã số doanh nghiệp/mã số hộ kinh doanh.
- Nếu có mã số chi nhánh dạng 10 chữ số, dấu "-", rồi 3 chữ số thì PHẢI lấy đầy đủ cả phần sau dấu "-".
- Ưu tiên mã số chi nhánh hơn mã số doanh nghiệp.
- Nếu không có mã số chi nhánh thì lấy mã số doanh nghiệp/mã số thuế.
- Không bỏ dấu "-".
- Không bỏ các số 0 sau dấu "-".

3b. QUY TẮC TÊN CÔNG TY

- ten_cong_ty là tên công ty/hộ kinh doanh, lấy theo loại giấy tờ:

- Nếu loai_giay_to = "HO_KINH_DOANH":
  + Lấy tên hộ kinh doanh, thường ở trường "Tên hộ kinh doanh".
  + Giữ nguyên dấu tiếng Việt, không viết tắt.

- Nếu loai_giay_to = "DOANH_NGHIEP":
  + Lấy tên doanh nghiệp bằng TIẾNG VIỆT, thường ở trường "Tên doanh nghiệp" hoặc "Tên công ty".
  + Nếu giấy tờ có cả tên tiếng Việt và tên viết bằng tiếng nước ngoài/tên viết tắt, CHỈ lấy tên tiếng Việt, KHÔNG lấy tên tiếng Anh hoặc tên viết tắt.
  + Giữ nguyên dấu tiếng Việt.

- Nếu không xác định được thì trả về null.

3c. QUY TẮC ĐỊA CHỈ TRỤ SỞ

- dia_chi_tru_so là địa chỉ trụ sở chính/địa chỉ trụ sở hộ kinh doanh, KHÔNG phải địa chỉ thường trú cá nhân trong CCCD.

- Nếu loai_giay_to = "HO_KINH_DOANH":
  + Lấy từ trường "Địa chỉ trụ sở hộ kinh doanh".

- Nếu loai_giay_to = "DOANH_NGHIEP":
  + Lấy từ trường "Địa chỉ trụ sở chính".

- Địa chỉ có thể xuống nhiều dòng:
→ phải nối đầy đủ thành một chuỗi duy nhất, giữ nguyên dấu tiếng Việt.

- KHÔNG lấy nhầm sang thuong_tru_cccd (địa chỉ thường trú cá nhân trên CCCD) hoặc địa chỉ liên lạc (nếu có).

- Nếu không có thì trả về null.

4. QUY TẮC NGƯỜI ĐẠI DIỆN

- nguoi_dai_dien là:
  + người đại diện theo pháp luật của doanh nghiệp
  + chủ hộ kinh doanh
  + hoặc chủ doanh nghiệp
  + Không lấy tên chủ thẻ CCCD làm người đại diện.

→ Phải lấy đúng tên người đại diện. Nếu không có người đại diện theo pháp luật thì lấy người đứng đầu.
- id_cccd_dai_dien là số CCCD/CMND/CMT/Hộ chiếu của người đại diện hoặc người đứng đầu.

5. QUY TẮC CHỨC DANH

- Nếu loai_giay_to = "HO_KINH_DOANH":
→ chuc_danh = "Chủ hộ kinh doanh".

- Nếu loai_giay_to = "DOANH_NGHIEP":
  + Ưu tiên lấy từ trường "Chức danh" hoặc "Chức vụ".
  + Nếu không có, lấy từ tiêu đề mục chứa tên người đại diện.

Ví dụ giá trị chức danh:
- "Giám đốc"
- "Tổng giám đốc"
- "Chủ doanh nghiệp"

6. QUY TẮC EMAIL & ĐIỆN THOẠI

- so_dien_thoai lấy từ các trường:
  + "Điện thoại"
  + "Số điện thoại"

- email lấy từ các trường:
  + "Email"
  + "Thư điện tử"

- Nếu không có thì trả về null.
"""

HINTS = {
    "CCCD": """
========================
GỢI Ý NHÓM FILE
========================

Các ảnh dưới đây ĐỀU LÀ CCCD/Căn cước (mặt trước và/hoặc mặt sau).
Chỉ điền id_cccd, ho_ten_cccd, thuong_tru_cccd.
Tất cả trường còn lại (kể cả loai_giay_to) = null.
""",
    "GPKD": """
========================
GỢI Ý NHÓM FILE
========================

Các ảnh dưới đây ĐỀU LÀ giấy chứng nhận đăng ký doanh nghiệp / hộ kinh doanh (có thể nhiều trang).
id_cccd, ho_ten_cccd, thuong_tru_cccd = null.
Số CCCD/CMND/Hộ chiếu của người đại diện ghi vào id_cccd_dai_dien.
""",
}


def build_prompt(group):
    return PROMPT_BOTH + HINTS[group]


print(len(build_prompt("CCCD")), "/", len(build_prompt("GPKD")), "ký tự")

## 7. Hàm đọc ảnh / PDF và gọi model

In [ ]:
import fitz  # PyMuPDF
from PIL import Image, ImageOps


def _prep(img):
    img = ImageOps.exif_transpose(img).convert("RGB")
    w, h = img.size
    scale = MAX_SIDE / max(w, h)
    if scale < 1:
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    return img


def load_images(path):
    "Ảnh -> [PIL]; PDF -> [PIL] cho tối đa MAX_PDF_PAGES trang đầu."
    if path.lower().endswith(".pdf"):
        doc = fitz.open(path)
        images = []
        for i in range(min(len(doc), MAX_PDF_PAGES)):
            pix = doc[i].get_pixmap(dpi=PDF_DPI)
            images.append(_prep(Image.frombytes("RGB", (pix.width, pix.height), pix.samples)))
        doc.close()
        return images
    with Image.open(path) as im:
        return [_prep(im)]


@torch.inference_mode()
def run_vlm(images, prompt, max_new_tokens=800):
    content = [{"type": "image", "image": im} for im in images]
    content.append({"type": "text", "text": prompt})
    messages = [{"role": "user", "content": content}]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    trimmed = out[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()


def parse_json(text):
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        return {}
    try:
        data = json.loads(m.group(0))
        return data if isinstance(data, dict) else {}
    except json.JSONDecodeError:
        return {}

## 8. Kiểm tra và chuẩn hoá từng trường

VLM có thể đọc nhầm số, nên mọi trường dạng số/định dạng đều được kiểm tra lại bằng code.

- `normalize(field, value)` trả về `(giá trị đã chuẩn hoá, lỗi)`; `lỗi = None` nghĩa là hợp lệ.
- `pick(field, candidates)` chọn **giá trị hợp lệ đầu tiên** trong các file (ví dụ CCCD mặt trước và mặt sau). Nếu không có giá trị nào hợp lệ: `STRICT=False` giữ giá trị đầu tiên kèm cảnh báo, `STRICT=True` đặt `null`.
- `id_cccd_dai_dien` chấp nhận cả số hộ chiếu (1 chữ cái + 7–8 số), không bị xoá mất chữ cái.

In [ ]:
LOAI_HOP_LE = {"DOANH_NGHIEP", "HO_KINH_DOANH"}
EMPTY_WORDS = {"", "null", "none", "n/a", "không có", "khong co"}

RE_MST = re.compile(r"\d{10}(-\d{3})?|\d{13}")
RE_PHONE = re.compile(r"(\+84|0)\d{8,10}")
RE_EMAIL = re.compile(r"[^@\s]+@[^@\s]+\.[^@\s]+")
RE_PASSPORT = re.compile(r"[A-Z]\d{7,8}")


def clean_value(v):
    if v is None:
        return None
    v = re.sub(r"\s+", " ", str(v)).strip()
    return None if v.lower() in EMPTY_WORDS else v


def normalize(field, v):
    "Chuẩn hoá một trường. Trả (giá_trị, lỗi); lỗi = None nếu hợp lệ."
    v = clean_value(v)
    if v is None:
        return None, None

    if field == "loai_giay_to":
        u = re.sub(r"[\s\-]+", "_", v).upper()
        if u in LOAI_HOP_LE:
            return u, None
        return None, f"giá trị không hợp lệ ({v!r}), đặt null"

    if field in ("id_cccd", "id_cccd_dai_dien"):
        compact = re.sub(r"[\s.\-]", "", v).upper()
        if field == "id_cccd_dai_dien" and RE_PASSPORT.fullmatch(compact):
            return compact, None  # số hộ chiếu
        digits = re.sub(r"\D", "", v)
        if len(digits) in (9, 12):
            return digits, None
        extra = ", hoặc số hộ chiếu" if field == "id_cccd_dai_dien" else ""
        return digits or v, f"kỳ vọng 12 số (CMND cũ 9 số{extra}) ({v!r})"

    if field == "ma_so_thue":
        v2 = re.sub(r"[\s.]", "", v)
        if RE_MST.fullmatch(v2):
            return v2, None
        return v2, f"kỳ vọng 10 hoặc 13 chữ số, hoặc dạng 10 số-3 số ({v!r})"

    if field == "so_dien_thoai":
        v2 = re.sub(r"[\s.\-()]", "", v)
        if RE_PHONE.fullmatch(v2):
            return v2, None
        return v2, f"không giống số điện thoại Việt Nam ({v!r})"

    if field == "email":
        v2 = v.lower().replace(" ", "")
        if RE_EMAIL.fullmatch(v2):
            return v2, None
        return v2, f"sai định dạng ({v!r})"

    return v, None


def pick(field, candidates, strict=None):
    "candidates: [(tên_file, giá_trị_thô), ...] theo thứ tự file. Trả (giá_trị, cảnh_báo)."
    strict = STRICT if strict is None else strict
    fallback, warns = None, []
    for name, raw in candidates:
        val, err = normalize(field, raw)
        if val is not None and err is None:
            return val, warns
        if err:
            warns.append(f"{name}: {field}: {err}")
        if fallback is None and val is not None:
            fallback = val
    if fallback is not None and not strict:
        return fallback, warns
    return None, warns

## 9. Pipeline: đọc theo từng nhóm rồi gộp kết quả

1. Với **mỗi file** trong nhóm, gọi model **một lần** bằng `PROMPT_BOTH` + gợi ý của nhóm đó.
2. **Chặn lẫn trường theo nhóm**: file trong `cccd_files` chỉ được lấy 3 trường CCCD; file trong `gpkd_files` chỉ được lấy `loai_giay_to` và các trường GPKD. Trường lạc nhóm bị bỏ và ghi cảnh báo (số CCCD in trong GPKD là của người đại diện, không bao giờ vào `id_cccd`).
3. Cảnh báo nếu một file không có trường nào của nhóm mình (có thể client gửi nhầm nhóm).
4. Với mỗi trường, chọn giá trị **hợp lệ** đầu tiên (đã kiểm tra định dạng **trước khi** gộp).
5. Hộ kinh doanh: `chuc_danh` luôn là `"Chủ hộ kinh doanh"`.

In [ ]:
import os


def extract_group(paths, group):
    "Gọi model cho từng file của một nhóm. Trả ({field: [(file, value), ...]}, cảnh_báo)."
    allowed = GROUP_FIELDS[group]
    others = [f for f in FIELDS if f not in allowed]
    prompt = build_prompt(group)
    cands = {k: [] for k in allowed}
    warns = []

    for p in paths:
        name = f"[{group}] {os.path.basename(p)}"
        try:
            images = load_images(p)
        except Exception as e:
            warns.append(f"{name}: không mở được file ({type(e).__name__})")
            continue
        if not images:
            warns.append(f"{name}: file không có trang nào")
            continue

        try:
            data = parse_json(run_vlm(images, prompt))
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            warns.append(f"{name}: hết bộ nhớ GPU, thử giảm MAX_SIDE / MAX_PDF_PAGES")
            continue
        if not data:
            warns.append(f"{name}: model không trả về JSON hợp lệ")
            continue

        leaked = [k for k in others if clean_value(data.get(k)) is not None]
        got = [k for k in allowed if clean_value(data.get(k)) is not None]
        if leaked:
            warns.append(f"{name}: bỏ {', '.join(leaked)} vì file thuộc nhóm {group}")
        if not got:
            hint = " (nội dung có vẻ thuộc nhóm khác, kiểm tra lại file)" if leaked else ""
            warns.append(f"{name}: không đọc được trường nào của nhóm {group}{hint}")
        for k in got:
            cands[k].append((name, data[k]))

    return cands, warns


def extract_files(cccd_paths=(), gpkd_paths=(), strict=None):
    "Đọc nhóm CCCD và nhóm GPKD rồi gộp thành một ExtractionResult."
    result = {k: None for k in FIELDS}
    warnings = []

    for group, paths in (("CCCD", cccd_paths), ("GPKD", gpkd_paths)):
        if not paths:
            continue
        cands, w = extract_group(list(paths), group)
        warnings += w
        for k, c in cands.items():
            result[k], w = pick(k, c, strict)
            warnings += w

    if result["loai_giay_to"] == "HO_KINH_DOANH":
        result["chuc_danh"] = "Chủ hộ kinh doanh"

    return ExtractionResult(**result), warnings

## 10. Chạy thử

Có **hai bước upload**:
1. Chọn file **CCCD** (mặt trước, mặt sau...). Không có CCCD thì bấm **Cancel**.
2. Chọn file **GPKD** (ảnh hoặc PDF, nhiều trang đều được). Không có GPKD thì bấm **Cancel**.

Chỉ một GPKD: bấm Cancel ở bước CCCD rồi chọn file GPKD.

In [ ]:
import shutil
from google.colab import files as colab_files

UPLOAD_DIR = "/content/uploads"


def upload_group(group):
    "Upload một nhóm file vào thư mục riêng (tránh trùng tên giữa hai nhóm)."
    print(f"\n>>> Chọn file {group} (bấm Cancel nếu không có):")
    try:
        uploaded = colab_files.upload()
    except Exception:
        uploaded = {}
    folder = os.path.join(UPLOAD_DIR, group.lower())
    shutil.rmtree(folder, ignore_errors=True)
    os.makedirs(folder)
    paths = []
    for name, data in (uploaded or {}).items():
        path = os.path.join(folder, os.path.basename(name))
        with open(path, "wb") as f:
            f.write(data)
        paths.append(path)
        if os.path.exists(name):
            os.remove(name)  # Colab tự lưu thêm một bản ở thư mục hiện tại
    return paths


cccd_paths = upload_group("CCCD")
gpkd_paths = upload_group("GPKD")
print("\nCCCD:", [os.path.basename(p) for p in cccd_paths])
print("GPKD:", [os.path.basename(p) for p in gpkd_paths])

In [ ]:
import time

assert cccd_paths or gpkd_paths, "Cần ít nhất một file CCCD hoặc GPKD"

t0 = time.time()
result, warnings = extract_files(cccd_paths, gpkd_paths)
print(f"Xong trong {time.time() - t0:.1f}s\n")

print(json.dumps(result.model_dump(), ensure_ascii=False, indent=2))
if warnings:
    print("\nCẢNH BÁO (nên kiểm tra thủ công):")
    for w in warnings:
        print(" -", w)

## 11. (Tuỳ chọn) API FastAPI `POST /ocr/both` ngay trong Colab

- `cccd_files` và `gpkd_files` đều **không bắt buộc**, nhưng phải có ít nhất một file (nếu không trả `400`).
- Lỗi: `400` file rỗng / sai định dạng / quá nhiều file, `413` file quá lớn.
- Phản hồi `200` đúng 12 trường của `ExtractionResult`. Số cảnh báo trả trong header `X-OCR-Warnings`; nội dung cảnh báo chỉ in ra log cell (không log dữ liệu trích xuất).
- Model chạy đồng bộ nên được đẩy sang thread (`asyncio.to_thread`) và khoá bằng `gpu_lock`: mỗi lúc chỉ một request dùng GPU, event loop không bị chặn.
- `custom_openapi` sinh schema OpenAPI 3.0.3 với `format: binary`, để Swagger (`/docs`) hiện nút chọn nhiều file cho cả hai trường.
- Trang `/` là form upload đơn giản để thử nhanh.

In [ ]:
import asyncio, tempfile, threading, time
from typing import List, Optional

import uvicorn
from fastapi import FastAPI, File, HTTPException, Response, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.openapi.utils import get_openapi
from fastapi.responses import HTMLResponse

app = FastAPI(title="FastAPI OCR API", version="0.1.0")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)
gpu_lock = asyncio.Lock()  # chỉ cho 1 request dùng GPU tại một thời điểm


async def save_group(files, group, tmp_dir):
    "Kiểm tra và lưu các file của một nhóm vào thư mục tạm. Trả danh sách đường dẫn."
    # Bỏ qua phần tử không có tên file (phòng client gửi file trống).
    files = [f for f in (files or []) if f is not None and f.filename]
    if len(files) > MAX_FILES_PER_GROUP:
        raise HTTPException(400, f"{group}_files: tối đa {MAX_FILES_PER_GROUP} file")

    paths = []
    for i, f in enumerate(files):
        ext = os.path.splitext(f.filename)[1].lower()
        if ext not in ALLOWED_EXT:
            raise HTTPException(400, f"{f.filename}: định dạng không hỗ trợ ({', '.join(sorted(ALLOWED_EXT))})")
        data = await f.read()
        if not data:
            raise HTTPException(400, f"{f.filename}: file rỗng")
        if len(data) > MAX_FILE_MB * 1024 * 1024:
            raise HTTPException(413, f"{f.filename}: vượt quá {MAX_FILE_MB}MB")
        path = os.path.join(tmp_dir, f"{group}_{i}{ext}")
        with open(path, "wb") as out:
            out.write(data)
        paths.append(path)
    return paths


@app.post("/ocr/both", response_model=ExtractionResult)
async def ocr_both(
    response: Response,
    cccd_files: Optional[List[UploadFile]] = File(None),
    gpkd_files: Optional[List[UploadFile]] = File(None),
):
    t0 = time.time()
    # Thư mục tạm tự xoá sau khi xử lý -> không lưu lại ảnh CCCD.
    with tempfile.TemporaryDirectory() as tmp_dir:
        cccd_paths = await save_group(cccd_files, "cccd", tmp_dir)
        gpkd_paths = await save_group(gpkd_files, "gpkd", tmp_dir)
        if not cccd_paths and not gpkd_paths:
            raise HTTPException(400, "Cần ít nhất một file trong cccd_files hoặc gpkd_files")

        async with gpu_lock:
            result, warnings = await asyncio.to_thread(extract_files, cccd_paths, gpkd_paths)

    response.headers["X-OCR-Warnings"] = str(len(warnings))
    # Chỉ log metadata, không log số CCCD / họ tên / địa chỉ.
    print(f"[ocr/both] cccd={len(cccd_paths)} gpkd={len(gpkd_paths)} "
          f"loai={result.loai_giay_to} warnings={len(warnings)} {time.time() - t0:.1f}s")
    return result


@app.get("/health")
async def health():
    return {"status": "ok", "model": MODEL_ID}


@app.get("/", response_class=HTMLResponse)
async def upload_page():
    return '''<!doctype html><html><head><meta charset="utf-8"><title>OCR CCCD / GPKD</title>
<style>body{font-family:sans-serif;max-width:760px;margin:2rem auto}label{display:block;margin:1rem 0}
pre{background:#f4f4f4;padding:1rem;white-space:pre-wrap}</style></head><body>
<h2>OCR CCCD / GPKD</h2>
<form id="f">
  <label>CCCD (mặt trước, mặt sau...): <input type="file" name="cccd_files" multiple accept="image/*,.pdf"></label>
  <label>GPKD (ảnh hoặc PDF): <input type="file" name="gpkd_files" multiple accept="image/*,.pdf"></label>
  <button type="submit">Trích xuất</button>
</form>
<pre id="out"></pre>
<script>
document.getElementById("f").onsubmit = async (e) => {
  e.preventDefault();
  const out = document.getElementById("out");
  out.textContent = "Đang xử lý...";
  // Chỉ gửi file thật sự được chọn: ô file trống sẽ bị FastAPI báo 422 (List[UploadFile]).
  const fd = new FormData();
  for (const inp of e.target.querySelectorAll("input[type=file]"))
    for (const file of inp.files) fd.append(inp.name, file);
  const r = await fetch("ocr/both", {method: "POST", body: fd});
  const body = await r.text();
  let text = body;
  try { text = JSON.stringify(JSON.parse(body), null, 2); } catch (_) {}
  out.textContent = "HTTP " + r.status + "  (cảnh báo: " + (r.headers.get("X-OCR-Warnings") || 0) + ")\\n" + text;
};
</script></body></html>'''


def custom_openapi():
    if app.openapi_schema:
        return app.openapi_schema

    openapi_schema = get_openapi(
        title="FastAPI OCR API",
        version="0.1.0",
        openapi_version="3.0.3",
        routes=app.routes,
    )

    body_schema = openapi_schema["components"]["schemas"].get(
        "Body_ocr_both_ocr_both_post"
    )

    if body_schema:
        # Make both fields optional in Swagger
        body_schema.pop("required", None)

        body_schema["properties"]["cccd_files"] = {
            "title": "Cccd Files",
            "type": "array",
            "items": {
                "type": "string",
                "format": "binary",
            },
        }

        body_schema["properties"]["gpkd_files"] = {
            "title": "Gpkd Files",
            "type": "array",
            "items": {
                "type": "string",
                "format": "binary",
            },
        }

    app.openapi_schema = openapi_schema
    return app.openapi_schema


app.openapi = custom_openapi

# Chạy lại cell này thì tắt server cũ trước để giải phóng port 8000.
if globals().get("server") is not None:
    server.should_exit = True
    time.sleep(2)

server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(2)
print("API đang chạy tại http://127.0.0.1:8000  (upload: /, Swagger: /docs)")

### Mở API từ trình duyệt của bạn

Colab chạy trên máy ảo riêng nên `127.0.0.1:8000` không mở trực tiếp được. Cell dưới tạo link proxy qua Colab: một link tới trang upload `/`, một link tới Swagger `/docs`.

In [ ]:
from google.colab.output import serve_kernel_port_as_window

serve_kernel_port_as_window(8000, path="/")       # trang upload
serve_kernel_port_as_window(8000, path="/docs")   # Swagger

### Gọi thử API bằng `requests`

Dùng lại các file đã upload ở mục 10, gửi CCCD vào `cccd_files` và GPKD vào `gpkd_files`.

In [ ]:
import requests

handles = []
try:
    form = []
    for field, paths in (("cccd_files", cccd_paths), ("gpkd_files", gpkd_paths)):
        for p in paths:
            h = open(p, "rb")
            handles.append(h)
            form.append((field, (os.path.basename(p), h)))
    r = requests.post("http://127.0.0.1:8000/ocr/both", files=form, timeout=600)
finally:
    for h in handles:
        h.close()

print("HTTP", r.status_code, "| X-OCR-Warnings:", r.headers.get("X-OCR-Warnings"))
print(json.dumps(r.json(), ensure_ascii=False, indent=2))

## Ghi chú

- **Hết bộ nhớ (OOM)**: giảm `MAX_SIDE` (ví dụ 1280) hoặc `MAX_PDF_PAGES`, hoặc đặt `LOAD_IN_4BIT = True`.
- **Đọc sai số / thiếu trường**: thử tăng `MAX_SIDE` lên 2000, chụp ảnh rõ và thẳng hơn, hoặc chỉnh prompt / `HINTS` ở mục 6.
- **Gửi nhầm nhóm** (ví dụ ảnh CCCD trong `gpkd_files`): kết quả nhóm đó gần như toàn `null` và có cảnh báo "nội dung có vẻ thuộc nhóm khác".
- **Muốn chạy production**: dùng vLLM làm server model riêng rồi cho FastAPI gọi qua API (xem file `.md` kế hoạch dự án). Khi đó chỉ cần thay `run_vlm` bằng hàm gọi vLLM; phần `extract_files`, `pick`, API và `custom_openapi` giữ nguyên.
- Luôn xem danh sách **CẢNH BÁO** và đối chiếu bằng mắt với các trường quan trọng (số CCCD, mã số thuế).